# Investment Screener
This notebook builds a simple, robust investment screener using the output of the Financial Ratio Engine (`data/processed/financial_ratios_generated.csv`). It tolerates missing columns and produces ranked outputs and preset lists.

In [12]:
import os
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

ratios_path = os.path.join("data", "processed", "financial_ratios_generated.csv")
companies_path = os.path.join("data", "processed", "companies_cleaned.csv")
sectors_path = os.path.join("data", "processed", "sectors_cleaned.csv")

print("Ratios file exists:", os.path.exists(ratios_path))
print("Companies file exists:", os.path.exists(companies_path))
print("Sectors file exists:", os.path.exists(sectors_path))

ratios = pd.read_csv(ratios_path) if os.path.exists(ratios_path) else pd.DataFrame()
print("Ratios shape:", ratios.shape)
ratios.head()

Ratios file exists: False
Companies file exists: False
Sectors file exists: False
Ratios shape: (0, 0)


""


In [13]:
# Prepare screener data and merge optional enrichment files
screener_data = ratios.copy()

if os.path.exists(companies_path):
    companies = pd.read_csv(companies_path)
    # normalize id column name if needed
    if 'id' in companies.columns and 'company_id' not in companies.columns:
        companies = companies.rename(columns={'id': 'company_id'})

    if 'company_id' in companies.columns and 'company_id' in screener_data.columns:
        companies['company_id'] = companies['company_id'].astype(str).str.strip().str.upper()
        screener_data['company_id'] = screener_data['company_id'].astype(str).str.strip().str.upper()
        keep = [c for c in ['company_id','company_name'] if c in companies.columns]
        screener_data = screener_data.merge(companies[keep], on='company_id', how='left')

if os.path.exists(sectors_path):
    sectors = pd.read_csv(sectors_path)
    # sectors file may have company_id column
    if 'company_id' not in sectors.columns and 'id' in sectors.columns:
        sectors = sectors.rename(columns={'id': 'company_id'})
    if 'company_id' in sectors.columns and 'company_id' in screener_data.columns:
        sectors['company_id'] = sectors['company_id'].astype(str).str.strip().str.upper()
        screener_data['company_id'] = screener_data['company_id'].astype(str).str.strip().str.upper()
        keep = [c for c in ['company_id','broad_sector','sub_sector'] if c in sectors.columns]
        screener_data = screener_data.merge(sectors[keep], on='company_id', how='left')

print('Screener data shape:', screener_data.shape)

Screener data shape: (0, 0)


In [14]:
# Ensure fiscal_year exists (extract 4-digit year from the 'year' column)
if 'fiscal_year' not in screener_data.columns:
    if 'year' in screener_data.columns:
        screener_data['fiscal_year'] = screener_data['year'].astype(str).str.extract(r"(\d{4})")[0]

screener_data['fiscal_year'] = pd.to_numeric(screener_data.get('fiscal_year'), errors='coerce')

# Keep latest available year per company
latest = (
    screener_data
    .dropna(subset=['fiscal_year'])
    .sort_values(['company_id','fiscal_year'])
    .groupby('company_id')
    .tail(1)
    .reset_index(drop=True)
)
print('Latest screener universe shape:', latest.shape)
latest.head()

KeyError: 'company_id'

In [15]:
# Scoring utilities

def percentile_score(series, higher_is_better=True):
    s = pd.to_numeric(series, errors='coerce')
    if higher_is_better:
        return s.rank(pct=True) * 100
    else:
        return (1 - s.rank(pct=True)) * 100

# Work with available column names in the generated CSV
scored = latest.copy()
# normalize CAGR column names from generator
if 'revenue_cagr_5yr' in scored.columns and 'revenue_cagr_5y_pct' not in scored.columns:
    scored = scored.rename(columns={'revenue_cagr_5yr': 'revenue_cagr_5y_pct'})
if 'pat_cagr_5yr' in scored.columns and 'pat_cagr_5y_pct' not in scored.columns:
    scored = scored.rename(columns={'pat_cagr_5yr': 'pat_cagr_5y_pct'})

# Define score mapping and compute (fall back to neutral 50 when missing)
score_columns = {
    'score_roe': ('return_on_equity_pct', True),
    'score_roce': ('return_on_capital_employed_pct', True),
    'score_margin': ('net_profit_margin_pct', True),
    'score_revenue_growth': ('revenue_cagr_5y_pct', True),
    'score_profit_growth': ('pat_cagr_5y_pct', True),
    'score_debt': ('debt_to_equity', False),
    'score_fcf': ('free_cash_flow_cr', True),
    'score_fcf_yield': ('fcf_yield_pct', True),
    'score_pe': ('pe_ratio', False),
    'score_pb': ('pb_ratio', False),
}

for score_name, (col, higher) in score_columns.items():
    if col in scored.columns:
        scored[score_name] = percentile_score(scored[col], higher)
    else:
        scored[score_name] = 50

# Composite score (weights can be adjusted)
scored['composite_score'] = (
    scored['score_roe'].fillna(50) * 0.15 +
    scored['score_roce'].fillna(50) * 0.15 +
    scored['score_margin'].fillna(50) * 0.1 +
    scored['score_revenue_growth'].fillna(50) * 0.15 +
    scored['score_profit_growth'].fillna(50) * 0.15 +
    scored['score_debt'].fillna(50) * 0.1 +
    scored['score_fcf'].fillna(50) * 0.1 +
    scored['score_pe'].fillna(50) * 0.05 +
    scored['score_pb'].fillna(50) * 0.05
)

scored = scored.sort_values('composite_score', ascending=False).reset_index(drop=True)
print('Scoring completed. Number of companies:', scored.shape[0])
scored.head()

NameError: name 'latest' is not defined

In [16]:
# Screener function that respects missing columns

def run_screener(
    data,
    sector=None,
    min_roe=None,
    min_roce=None,
    max_debt_to_equity=None,
    min_fcf=None,
    min_revenue_cagr_5y=None,
    min_pat_cagr_5y=None,
    max_pe=None,
    max_pb=None,
    min_net_profit_margin=None,
    debt_free_only=False,
    fcf_positive_only=False,
    min_composite_score=None,
    top_n=50
):
    result = data.copy()
    if sector is not None and 'broad_sector' in result.columns:
        result = result[result['broad_sector'] == sector]
    if min_roe is not None and 'return_on_equity_pct' in result.columns:
        result = result[result['return_on_equity_pct'] >= min_roe]
    if min_roce is not None and 'return_on_capital_employed_pct' in result.columns:
        result = result[result['return_on_capital_employed_pct'] >= min_roce]
    if max_debt_to_equity is not None and 'debt_to_equity' in result.columns:
        result = result[result['debt_to_equity'] <= max_debt_to_equity]
    if min_fcf is not None and 'free_cash_flow_cr' in result.columns:
        result = result[result['free_cash_flow_cr'] >= min_fcf]
    if min_revenue_cagr_5y is not None and 'revenue_cagr_5y_pct' in result.columns:
        result = result[result['revenue_cagr_5y_pct'] >= min_revenue_cagr_5y]
    if min_pat_cagr_5y is not None and 'pat_cagr_5y_pct' in result.columns:
        result = result[result['pat_cagr_5y_pct'] >= min_pat_cagr_5y]
    if max_pe is not None and 'pe_ratio' in result.columns:
        result = result[result['pe_ratio'] <= max_pe]
    if max_pb is not None and 'pb_ratio' in result.columns:
        result = result[result['pb_ratio'] <= max_pb]
    if min_net_profit_margin is not None and 'net_profit_margin_pct' in result.columns:
        result = result[result['net_profit_margin_pct'] >= min_net_profit_margin]
    if debt_free_only and 'debt_free_flag' in result.columns:
        result = result[result['debt_free_flag'] == True]
    if fcf_positive_only and 'fcf_positive_flag' in result.columns:
        result = result[result['fcf_positive_flag'] == True]
    if min_composite_score is not None:
        result = result[result['composite_score'] >= min_composite_score]

    return result.sort_values('composite_score', ascending=False).head(top_n)

In [17]:
# Preset screeners
preset_results = {}

preset_results['quality_companies'] = run_screener(scored, min_roe=15, min_roce=15, max_debt_to_equity=1, min_fcf=0, top_n=50)
preset_results['value_companies'] = run_screener(scored, max_pe=25, max_pb=4, min_roe=10, top_n=50)
preset_results['growth_companies'] = run_screener(scored, min_revenue_cagr_5y=10, min_pat_cagr_5y=10, min_roe=10, top_n=50)
preset_results['debt_free_companies'] = run_screener(scored, debt_free_only=True, min_roe=10, fcf_positive_only=True, top_n=50)

preset_summary = pd.DataFrame({'preset_name': list(preset_results.keys()), 'company_count': [len(df) for df in preset_results.values()]})
print('Preset summary:')
print(preset_summary)

NameError: name 'scored' is not defined

In [18]:
# Save outputs
os.makedirs('output', exist_ok=True)
scored.to_csv(os.path.join('output','screener_full_ranked_universe.csv'), index=False)
preset_summary.to_csv(os.path.join('output','screener_presets_summary.csv'), index=False)
print('Saved outputs to output/ directory')

NameError: name 'scored' is not defined

Notebook rebuilt and saved. Next steps: run this notebook to generate the updated screener outputs, or tell me if you want the notebook adapted into a Streamlit page under `src/dashboard/pages/`. 

<a href="https://colab.research.google.com/github/gundalarakesh262-cpu/nifty100-financial-intelligence/blob/main/notebooks/Investment_Screener.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import os
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

ratios_path = os.path.join("data", "processed", "financial_ratios_generated.csv")
companies_path = os.path.join("data", "processed", "companies_cleaned.csv")
sectors_path = os.path.join("data", "processed", "sectors_cleaned.csv")

print("Ratios file exists:", os.path.exists(ratios_path))
print("Companies file exists:", os.path.exists(companies_path))
print("Sectors file exists:", os.path.exists(sectors_path))

ratios = pd.read_csv(ratios_path)

print("Ratios shape:", ratios.shape)
ratios.head()

Ratios file exists: False
Companies file exists: False
Sectors file exists: False


FileNotFoundError: [Errno 2] No such file or directory: 'data\\processed\\financial_ratios_generated.csv'

In [3]:
screener_data = ratios.copy()

# Add company names if companies file exists
if os.path.exists(companies_path):
    companies = pd.read_csv(companies_path)

    if "id" in companies.columns:
        companies["id"] = companies["id"].astype(str).str.strip().str.upper()
        screener_data["company_id"] = screener_data["company_id"].astype(str).str.strip().str.upper()

        keep_cols = [col for col in ["id", "company_name"] if col in companies.columns]

        screener_data = screener_data.merge(
            companies[keep_cols],
            left_on="company_id",
            right_on="id",
            how="left"
        )

# Add sectors if sectors file exists
if os.path.exists(sectors_path):
    sectors = pd.read_csv(sectors_path)

    sectors["company_id"] = sectors["company_id"].astype(str).str.strip().str.upper()
    screener_data["company_id"] = screener_data["company_id"].astype(str).str.strip().str.upper()

    keep_cols = [col for col in ["company_id", "broad_sector", "sub_sector"] if col in sectors.columns]

    screener_data = screener_data.merge(
        sectors[keep_cols],
        on="company_id",
        how="left"
    )

print("Screener data shape:", screener_data.shape)
screener_data.head()

NameError: name 'ratios' is not defined

In [10]:
if "fiscal_year" not in screener_data.columns:
    screener_data["fiscal_year"] = screener_data["year"].astype(str).str.extract(r"(\d{4})")[0]

if "fiscal_year" not in screener_data.columns:
    screener_data["fiscal_year"] = (
        screener_data["year"].astype(str).str.extract(r"(\d{4})")[0]
    )

if "fiscal_year" not in screener_data.columns:
    screener_data["fiscal_year"] = screener_data["year"].astype(str).str.extract(r"(\d{4})")[0]

screener_data["fiscal_year"] = pd.to_numeric(screener_data["fiscal_year"], errors="coerce")

latest = (
    screener_data
    .dropna(subset=["fiscal_year"])
    .sort_values(["company_id", "fiscal_year"])
    .groupby("company_id")
    .tail(1)
    .reset_index(drop=True)
)

print("Latest screener universe:", latest.shape)
print("Unique companies:", latest["company_id"].nunique())

preview_cols = ["company_id", "company_name", "broad_sector", "year", "fiscal_year"]

preview_cols = [col for col in preview_cols if col in latest.columns]latest[preview_cols].head()


SyntaxError: invalid syntax (199008881.py, line 28)

In [ ]:
def percentile_score(series, higher_is_better=True):
    s = pd.to_numeric(series, errors="coerce")

    if higher_is_better:
        return s.rank(pct=True) * 100
    else:
        return (1 - s.rank(pct=True)) * 100


scored = latest.copy()

# Normalize generated ratio column names for scoring
if "revenue_cagr_5yr" in scored.columns and "revenue_cagr_5y_pct" not in scored.columns:
    scored = scored.rename(columns={"revenue_cagr_5yr": "revenue_cagr_5y_pct"})
if "pat_cagr_5yr" in scored.columns and "pat_cagr_5y_pct" not in scored.columns:
    scored = scored.rename(columns={"pat_cagr_5yr": "pat_cagr_5y_pct"})

score_columns = {
    "score_roe": ("return_on_equity_pct", True),
    "score_roce": ("return_on_capital_employed_pct", True),
    "score_margin": ("net_profit_margin_pct", True),
    "score_revenue_growth": ("revenue_cagr_5y_pct", True),
    "score_profit_growth": ("pat_cagr_5y_pct", True),
    "score_debt": ("debt_to_equity", False),
    "score_fcf": ("free_cash_flow_cr", True),
    "score_fcf_yield": ("fcf_yield_pct", True),
    "score_pe": ("pe_ratio", False),
    "score_pb": ("pb_ratio", False),
}

for score_name, (col_name, higher_is_better) in score_columns.items():
    if col_name in scored.columns:
        scored[score_name] = percentile_score(scored[col_name], higher_is_better)
    else:
        scored[score_name] = 50

# Final composite score
scored["composite_score"] = (
    scored["score_roe"] * 0.15 +
    scored["score_roce"] * 0.15 +
    scored["score_margin"] * 0.10 +
    scored["score_revenue_growth"] * 0.15 +
    scored["score_profit_growth"] * 0.15 +
    scored["score_debt"] * 0.10 +
    scored["score_fcf"] * 0.10 +
    scored["score_pe"] * 0.05 +
    scored["score_pb"] * 0.05
)

scored = scored.sort_values("composite_score", ascending=False)

print("Scoring completed.")
score_preview_cols = [
    "company_id",
    "company_name",
    "broad_sector",
    "composite_score",
    "return_on_equity_pct",
    "return_on_capital_employed_pct",
    "debt_to_equity",
    "free_cash_flow_cr",
    "revenue_cagr_5y_pct",
    "pat_cagr_5y_pct",
    "pe_ratio",
    "pb_ratio"

]scored[score_preview_cols].head(20)


score_preview_cols = [col for col in score_preview_cols if col in scored.columns]

Scoring completed.


,company_id,company_name,broad_sector,composite_score,return_on_equity_pct,return_on_capital_employed_pct,debt_to_equity,free_cash_flow_cr,revenue_cagr_5y_pct,pat_cagr_5y_pct,pe_ratio,pb_ratio
46,INDIGO,Interglobe Aviation Ltd,Consumer Discretionary,79.974105,892.568306,4953.540773,0.018579,9424.0,19.313355,120.692509,76.23,7.87
50,IRCTC,Indian Railway Catering & Tourism Corporation Ltd,Consumer Discretionary,75.169912,34.396285,42.826748,0.018576,667.0,17.955244,29.166864,60.44,5.74
89,TRENT,Trent Ltd,Consumer Discretionary,71.347710,36.307768,22.332932,0.430924,841.0,36.306916,73.113676,8.65,11.82
61,LTIM,LTIMindtree Ltd,Information Technology,70.609001,22.904386,25.207117,0.103457,1752.0,30.327981,24.775129,64.52,2.38
35,HAL,Hindustan Aeronautics Ltd,Industrials,69.373699,3816.582915,2590.993789,0.618090,1814.0,8.712549,26.485271,71.89,3.65
5,ADANIPOWER,Adani Power Ltd,Energy,66.668946,48.276741,18.385823,0.802318,17651.0,16.086096,NaN,44.57,9.91
48,INFY,Infosys Ltd,Information Technology,65.773503,29.788007,32.906971,0.094864,20117.0,13.199101,11.239420,35.99,8.97
66,NESTLEIND,Nestle India Ltd,Consumer Staples,65.391557,117.754491,143.147897,0.103293,2938.0,14.548574,14.852320,54.96,14.34
58,LICI,Life Insurance Corporation of India,Financials,65.258919,49.447110,39.050358,0.000000,853.0,8.159861,73.175567,30.67,8.10
52,ITC,ITC Ltd\n,Consumer Staples,65.204650,27.851074,32.638685,0.004067,18742.0,7.950897,10.083408,47.85,3.24


In [ ]:
def run_screener(
    data,
    sector=None,
    min_roe=None,
    min_roce=None,
    max_debt_to_equity=None,
    min_fcf=None,
    min_revenue_cagr_5y=None,
    min_pat_cagr_5y=None,
    max_pe=None,
    max_pb=None,
    min_net_profit_margin=None,
    debt_free_only=False,
    fcf_positive_only=False,
    min_composite_score=None,
    top_n=50
):
    result = data.copy()

    if sector is not None and "broad_sector" in result.columns:
        result = result[result["broad_sector"] == sector]

    if min_roe is not None:
        result = result[result["return_on_equity_pct"] >= min_roe]

    if min_roce is not None and "return_on_capital_employed_pct" in result.columns:
        result = result[result["return_on_capital_employed_pct"] >= min_roce]

    if max_debt_to_equity is not None:
        result = result[result["debt_to_equity"] <= max_debt_to_equity]

    if min_fcf is not None:
        result = result[result["free_cash_flow_cr"] >= min_fcf]

    if min_revenue_cagr_5y is not None and "revenue_cagr_5y_pct" in result.columns:
        result = result[result["revenue_cagr_5y_pct"] >= min_revenue_cagr_5y]

    if min_pat_cagr_5y is not None and "pat_cagr_5y_pct" in result.columns:
        result = result[result["pat_cagr_5y_pct"] >= min_pat_cagr_5y]

    if max_pe is not None and "pe_ratio" in result.columns:
        result = result[result["pe_ratio"] <= max_pe]

    if max_pb is not None and "pb_ratio" in result.columns:
        result = result[result["pb_ratio"] <= max_pb]

    if min_net_profit_margin is not None:
        result = result[result["net_profit_margin_pct"] >= min_net_profit_margin]

    if debt_free_only and "debt_free_flag" in result.columns:
        result = result[result["debt_free_flag"] == True]

    if fcf_positive_only and "fcf_positive_flag" in result.columns:
        result = result[result["fcf_positive_flag"] == True]

    if min_composite_score is not None:
        result = result[result["composite_score"] >= min_composite_score]

    return result.sort_values("composite_score", ascending=False).head(top_n)

In [6]:
preset_results = {}

preset_results["quality_companies"] = run_screener(
    scored,
    min_roe=15,
    min_roce=15,
    max_debt_to_equity=1,
    min_fcf=0,
    top_n=50
)

preset_results["value_companies"] = run_screener(
    scored,
    max_pe=25,
    max_pb=4,
    min_roe=10,
    top_n=50
)

preset_results["growth_companies"] = run_screener(
    scored,
    min_revenue_cagr_5y=10,
    min_pat_cagr_5y=10,
    min_roe=10,
    top_n=50
)

preset_results["debt_free_companies"] = run_screener(
    scored,
    debt_free_only=True,
    min_roe=10,
    fcf_positive_only=True,
    top_n=50
)

preset_results["cash_flow_kings"] = run_screener(
    scored,
    min_fcf=0,
    fcf_positive_only=True,
    min_net_profit_margin=10,
    top_n=50
)

preset_results["balanced_winners"] = run_screener(
    scored,
    min_roe=12,
    min_roce=12,
    max_debt_to_equity=1.5,
    min_revenue_cagr_5y=8,
    min_composite_score=60,
    top_n=50
)

preset_summary = pd.DataFrame({
    "preset_name": list(preset_results.keys()),
    "company_count": [len(df) for df in preset_results.values()]
})

preset_summary

NameError: name 'scored' is not defined

In [7]:
display_cols = [
    "company_id",
    "company_name",
    "broad_sector",
    "year",
    "fiscal_year",
    "composite_score",
    "return_on_equity_pct",
    "return_on_capital_employed_pct",
    "net_profit_margin_pct",
    "debt_to_equity",
    "free_cash_flow_cr",
    "revenue_cagr_5y_pct",
    "pat_cagr_5y_pct",
    "pe_ratio",
    "pb_ratio",
    "capital_allocation_pattern"
]

# Keep only columns that exist
display_cols = [col for col in display_cols if col in scored.columns]

preset_results["quality_companies"][display_cols].head(20)

NameError: name 'scored' is not defined

In [ ]:
# Save outputs in current Colab folder

scored.to_csv("screener_full_ranked_universe.csv", index=False)
preset_summary.to_csv("screener_presets_summary.csv", index=False)

excel_path = "screener_output.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    scored[display_cols].to_excel(writer, sheet_name="Full Ranked Universe", index=False)

    for name, result in preset_results.items():
        result[display_cols].to_excel(writer, sheet_name=name[:31], index=False)

print("Investment Screener outputs created successfully!")
print("Saved files:")
print("screener_full_ranked_universe.csv")
print("screener_presets_summary.csv")
print("screener_output.xlsx")

Investment Screener outputs created successfully!
Saved files:
screener_full_ranked_universe.csv
screener_presets_summary.csv
screener_output.xlsx


In [30]:
# ---------------------------------------------------------
# Outlier-adjusted scoring improvement
# Caps extreme values so one unusual ROE/ROCE does not dominate rankings
# ---------------------------------------------------------

def cap_outliers(series, lower=0.01, upper=0.99):
    s = pd.to_numeric(series, errors="coerce")
    lower_limit = s.quantile(lower)
    upper_limit = s.quantile(upper)
    return s.clip(lower=lower_limit, upper=upper_limit)


score_input_cols = [
    "return_on_equity_pct",
    "return_on_capital_employed_pct",
    "net_profit_margin_pct",
    "revenue_cagr_5y_pct",
    "pat_cagr_5y_pct",
    "debt_to_equity",
    "free_cash_flow_cr",
    "fcf_yield_pct",
    "pe_ratio",
    "pb_ratio"
]

for col in score_input_cols:
    if col in scored.columns:
        scored[f"{col}_capped"] = cap_outliers(scored[col])


# Recalculate percentile scores using capped values
scored["score_roe"] = percentile_score(scored["return_on_equity_pct_capped"], True)
scored["score_roce"] = percentile_score(scored["return_on_capital_employed_pct_capped"], True)
scored["score_margin"] = percentile_score(scored["net_profit_margin_pct_capped"], True)

scored["score_revenue_growth"] = percentile_score(scored["revenue_cagr_5y_pct_capped"], True)
scored["score_profit_growth"] = percentile_score(scored["pat_cagr_5y_pct_capped"], True)

scored["score_debt"] = percentile_score(scored["debt_to_equity_capped"], False)

scored["score_fcf"] = percentile_score(scored["free_cash_flow_cr_capped"], True)
scored["score_fcf_yield"] = percentile_score(scored["fcf_yield_pct_capped"], True)

scored["score_pe"] = percentile_score(scored["pe_ratio_capped"], False)
scored["score_pb"] = percentile_score(scored["pb_ratio_capped"], False)

scored["composite_score"] = (
    scored["score_roe"].fillna(50) * 0.15 +
    scored["score_roce"].fillna(50) * 0.15 +
    scored["score_margin"].fillna(50) * 0.10 +
    scored["score_revenue_growth"].fillna(50) * 0.15 +
    scored["score_profit_growth"].fillna(50) * 0.15 +
    scored["score_debt"].fillna(50) * 0.10 +
    scored["score_fcf"].fillna(50) * 0.10 +
    scored["score_pe"].fillna(50) * 0.05 +
    scored["score_pb"].fillna(50) * 0.05
)

scored = scored.sort_values("composite_score", ascending=False)

print("Outlier-adjusted composite score recalculated.")
scored[display_cols].head(10)

Outlier-adjusted composite score recalculated.


,company_id,company_name,broad_sector,year,fiscal_year,composite_score,return_on_equity_pct,return_on_capital_employed_pct,net_profit_margin_pct,debt_to_equity,free_cash_flow_cr,revenue_cagr_5y_pct,pat_cagr_5y_pct,pe_ratio,pb_ratio,capital_allocation_pattern
46,INDIGO,Interglobe Aviation Ltd,Consumer Discretionary,Mar 2024,2024,79.974105,892.568306,4953.540773,11.852723,0.018579,9424.0,19.313355,120.692509,76.23,7.87,Reinvesting + Returning Capital
50,IRCTC,Indian Railway Catering & Tourism Corporation Ltd,Consumer Discretionary,Mar 2024,2024,75.169912,34.396285,42.826748,26.018735,0.018576,667.0,17.955244,29.166864,60.44,5.74,Reinvesting + Returning Capital
89,TRENT,Trent Ltd,Consumer Discretionary,Mar 2024,2024,71.347710,36.307768,22.332932,11.935354,0.430924,841.0,36.306916,73.113676,8.65,11.82,Reinvesting + Returning Capital
61,LTIM,LTIMindtree Ltd,Information Technology,Mar 2024,2024,70.609001,22.904386,25.207117,12.909311,0.103457,1752.0,30.327981,24.775129,64.52,2.38,Reinvesting + Returning Capital
35,HAL,Hindustan Aeronautics Ltd,Industrials,Mar 2024,2024,69.373699,3816.582915,2590.993789,24.999177,0.618090,1814.0,8.712549,26.485271,71.89,3.65,Reinvesting + Returning Capital
5,ADANIPOWER,Adani Power Ltd,Energy,Mar 2024,2024,66.668946,48.276741,18.385823,41.367599,0.802318,17651.0,16.086096,NaN,44.57,9.91,Asset Sale + Debt/Dividend Outflow
48,INFY,Infosys Ltd,Information Technology,Mar 2024,2024,65.773503,29.788007,32.906971,17.080757,0.094864,20117.0,13.199101,11.239420,35.99,8.97,Reinvesting + Returning Capital
66,NESTLEIND,Nestle India Ltd,Consumer Staples,Mar 2024,2024,65.391557,117.754491,143.147897,16.122817,0.103293,2938.0,14.548574,14.852320,54.96,14.34,Reinvesting + Returning Capital
58,LICI,Life Insurance Corporation of India,Financials,Mar 2024,2024,65.258919,49.447110,39.050358,4.836601,0.000000,853.0,8.159861,73.175567,30.67,8.10,Reinvesting + Returning Capital
52,ITC,ITC Ltd\n,Consumer Staples,Mar 2024,2024,65.204650,27.851074,32.638685,29.282025,0.004067,18742.0,7.950897,10.083408,47.85,3.24,Asset Sale + Debt/Dividend Outflow


In [31]:
print("===== INVESTMENT SCREENER VALIDATION =====")

print("\nLatest universe shape:")
print(latest.shape)

print("\nUnique companies in latest universe:")
print(latest["company_id"].nunique())

print("\nScored universe shape:")
print(scored.shape)

print("\nDuplicate companies in latest universe:")
print(latest.duplicated(["company_id"]).sum())

print("\nPreset summary:")
display(preset_summary)

print("\nOutput files created:")
import os
print("screener_full_ranked_universe.csv:", os.path.exists("screener_full_ranked_universe.csv"))
print("screener_presets_summary.csv:", os.path.exists("screener_presets_summary.csv"))
print("screener_output.xlsx:", os.path.exists("screener_output.xlsx"))

print("\nTop 10 companies by composite score:")
display(scored[display_cols].head(10))

===== INVESTMENT SCREENER VALIDATION =====

Latest universe shape:
(98, 66)

Unique companies in latest universe:
98

Scored universe shape:
(98, 87)

Duplicate companies in latest universe:
0

Preset summary:


,preset_name,company_count
0,quality_companies,36
1,value_companies,5
2,growth_companies,41
3,debt_free_companies,1
4,cash_flow_kings,43
5,balanced_winners,18



Output files created:
screener_full_ranked_universe.csv: True
screener_presets_summary.csv: True
screener_output.xlsx: True

Top 10 companies by composite score:


,company_id,company_name,broad_sector,year,fiscal_year,composite_score,return_on_equity_pct,return_on_capital_employed_pct,net_profit_margin_pct,debt_to_equity,free_cash_flow_cr,revenue_cagr_5y_pct,pat_cagr_5y_pct,pe_ratio,pb_ratio,capital_allocation_pattern
46,INDIGO,Interglobe Aviation Ltd,Consumer Discretionary,Mar 2024,2024,79.974105,892.568306,4953.540773,11.852723,0.018579,9424.0,19.313355,120.692509,76.23,7.87,Reinvesting + Returning Capital
50,IRCTC,Indian Railway Catering & Tourism Corporation Ltd,Consumer Discretionary,Mar 2024,2024,75.169912,34.396285,42.826748,26.018735,0.018576,667.0,17.955244,29.166864,60.44,5.74,Reinvesting + Returning Capital
89,TRENT,Trent Ltd,Consumer Discretionary,Mar 2024,2024,71.347710,36.307768,22.332932,11.935354,0.430924,841.0,36.306916,73.113676,8.65,11.82,Reinvesting + Returning Capital
61,LTIM,LTIMindtree Ltd,Information Technology,Mar 2024,2024,70.609001,22.904386,25.207117,12.909311,0.103457,1752.0,30.327981,24.775129,64.52,2.38,Reinvesting + Returning Capital
35,HAL,Hindustan Aeronautics Ltd,Industrials,Mar 2024,2024,69.373699,3816.582915,2590.993789,24.999177,0.618090,1814.0,8.712549,26.485271,71.89,3.65,Reinvesting + Returning Capital
5,ADANIPOWER,Adani Power Ltd,Energy,Mar 2024,2024,66.668946,48.276741,18.385823,41.367599,0.802318,17651.0,16.086096,NaN,44.57,9.91,Asset Sale + Debt/Dividend Outflow
48,INFY,Infosys Ltd,Information Technology,Mar 2024,2024,65.773503,29.788007,32.906971,17.080757,0.094864,20117.0,13.199101,11.239420,35.99,8.97,Reinvesting + Returning Capital
66,NESTLEIND,Nestle India Ltd,Consumer Staples,Mar 2024,2024,65.391557,117.754491,143.147897,16.122817,0.103293,2938.0,14.548574,14.852320,54.96,14.34,Reinvesting + Returning Capital
58,LICI,Life Insurance Corporation of India,Financials,Mar 2024,2024,65.258919,49.447110,39.050358,4.836601,0.000000,853.0,8.159861,73.175567,30.67,8.10,Reinvesting + Returning Capital
52,ITC,ITC Ltd\n,Consumer Staples,Mar 2024,2024,65.204650,27.851074,32.638685,29.282025,0.004067,18742.0,7.950897,10.083408,47.85,3.24,Asset Sale + Debt/Dividend Outflow


## Investment Screener — Module 3

This module builds an investment screening engine using the output of the Financial Ratio Engine.

### Input Used
- `financial_ratios_generated.csv`
- Optional enrichment:
  - `companies_cleaned.csv`
  - `sectors_cleaned.csv`

### Main Logic
The screener uses the latest available financial year for each company and applies both preset and custom filters.

### Scoring Method
Each company receives a composite score based on:
- ROE
- ROCE
- Net profit margin
- Revenue CAGR
- PAT CAGR
- Debt-to-equity
- Free cash flow
- FCF yield
- P/E ratio
- P/B ratio

Higher profitability, higher growth, stronger cash flow, lower debt, and cheaper valuation improve the score.

### Preset Screeners Created
1. Quality Companies
2. Value Companies
3. Growth Companies
4. Debt-Free Companies
5. Cash Flow Kings
6. Balanced Winners

### Outputs
- `screener_full_ranked_universe.csv`
- `screener_presets_summary.csv`
- `screener_output.xlsx`

### Purpose
This screener helps analysts quickly shortlist Nifty 100 companies based on financial quality, valuation, growth, leverage, and cash-flow strength.

In [ ]:
from google.colab import files

files.download("screener_full_ranked_universe.csv")
files.download("screener_presets_summary.csv")
files.download("screener_output.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>